In [14]:
import pandas as pd
import numpy as np
import seaborn as sns 
import matplotlib.pyplot as plt
import plotly.express as px
import category_encoders as ce
from sklearn.preprocessing import StandardScaler
from pathlib import Path 

pd.set_option('display.max_columns', None) #to show all columns when we print the dataframe

In [15]:
#Read the csv file
data_path = Path("/mnt/e/Users/KilianAT/Documents/Estudios/Weiterbildung/Data Science/Proyecto/mle_liora_london_firefighter/data")

df = pd.read_csv(data_path / "dataset_with_filtered_distance_speed.csv")

print(df.columns.tolist())


['index', 'IncidentNumber', 'CalYear', 'Month', 'Weekday', 'Hour', 'Is_Nightshift', 'Is_Rush_Hour', 'Is_Weekend', 'Is_Public_Holiday', 'AttendanceTimeSeconds', 'DeployedFromStation_Name', 'PumpOrder', 'IncidentGroup', 'Is_SpecialService', 'SpecialServiceType', 'PropertyCategory', 'PropertyType', 'IncGeo_BoroughName', 'NumCalls', 'Is_RepeatedCall', 'Latitude', 'Longitude', 'distance_fire_to_station', 'avg_speed', 'NumOfCalls_bucket']


In [16]:
df = pd.read_csv(data_path / "dataset_with_filtered_distance_speed.csv")
print(df.groupby('CalYear').size())

CalYear
2021     93601
2022    107867
2023    109192
2024    116221
2025    119150
2026     18675
dtype: int64


In [17]:
#UNCHANGED
# Number of calls, outlier management - buckets 
def bucket_num_calls(x):
    if x <= 1:
        return "1"
    elif x == 2:
        return "2"
    elif x == 3:
        return "3"
    elif x <= 5:
        return "4-5"
    elif x <= 10:
        return "6-10"
    else:
        return "10+"

col = "NumCalls"  
df[col] = pd.to_numeric(df[col], errors='coerce').fillna(1)
df["NumOfCalls_bucket"] = df[col].apply(bucket_num_calls)

In [18]:
#UNCHANGED
# Delete Column NumCalls
df = df.drop(columns=["NumCalls"])

In [19]:
#UNCHANGED
# Is_central_London as new column
inner_london = [
    'CITY OF LONDON', 'WESTMINSTER', 'CAMDEN', 'ISLINGTON', 'HACKNEY', 
    'TOWER HAMLETS', 'SOUTHWARK', 'LAMBETH', 'KENSINGTON AND CHELSEA', 
    'HAMMERSMITH AND FULHAM', 'WANDSWORTH', 'LEWISHAM', 'NEWHAM', 'HARINGEY'
]
df['Is_central_London'] = df['IncGeo_BoroughName'].isin(inner_london).astype(int)

In [20]:
#CHANGED
#Save pre-encoding DataFrame
df.to_csv(data_path / "dataset_pre_encoding.csv", index=False)
print(f"Saved! Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Saved! Shape: (564706, 26)
Columns: ['index', 'IncidentNumber', 'CalYear', 'Month', 'Weekday', 'Hour', 'Is_Nightshift', 'Is_Rush_Hour', 'Is_Weekend', 'Is_Public_Holiday', 'AttendanceTimeSeconds', 'DeployedFromStation_Name', 'PumpOrder', 'IncidentGroup', 'Is_SpecialService', 'SpecialServiceType', 'PropertyCategory', 'PropertyType', 'IncGeo_BoroughName', 'Is_RepeatedCall', 'Latitude', 'Longitude', 'distance_fire_to_station', 'avg_speed', 'NumOfCalls_bucket', 'Is_central_London']


In [21]:
# CHANGED
# CHANGE: split changed from single train/test (SPLIT_YEAR=2025)
#         to chronological 70/15/15 (train≤2023, test=2024, val≥2025)

TARGET   = "AttendanceTimeSeconds"
DROP_COLS = ["index", "IncidentNumber", "avg_speed"]

# CHANGED: three splits instead of two
train_df = df[df["CalYear"] <= 2023].copy()
test_df  = df[df["CalYear"] == 2024].copy()
val_df   = df[df["CalYear"] >= 2025].copy()

n = len(df)
print(f"Train: {len(train_df)} ({len(train_df)/n*100:.1f}%) | Years: 2021-2023")
print(f"Test:  {len(test_df)} ({len(test_df)/n*100:.1f}%) | Years: 2024")
print(f"Val:   {len(val_df)} ({len(val_df)/n*100:.1f}%) | Years: 2025-2026")

def get_baseline_data(data):
    to_drop = [TARGET] + DROP_COLS
    X = data.drop(columns=[c for c in to_drop if c in data.columns])
    y = data[TARGET]
    y_log = np.log1p(y)
    return X, y, y_log

X_train, y_train, y_train_log = get_baseline_data(train_df)
X_test,  y_test,  y_test_log  = get_baseline_data(test_df)
# CHANGED: added validation set
X_val,   y_val,   y_val_log   = get_baseline_data(val_df)

print(f"\nFeatures in model: {list(X_train.columns)}")
print(f"Train: {len(X_train)} | Test: {len(X_test)} | Val: {len(X_val)}")

Train: 310660 (55.0%) | Years: 2021-2023
Test:  116221 (20.6%) | Years: 2024
Val:   137825 (24.4%) | Years: 2025-2026

Features in model: ['CalYear', 'Month', 'Weekday', 'Hour', 'Is_Nightshift', 'Is_Rush_Hour', 'Is_Weekend', 'Is_Public_Holiday', 'DeployedFromStation_Name', 'PumpOrder', 'IncidentGroup', 'Is_SpecialService', 'SpecialServiceType', 'PropertyCategory', 'PropertyType', 'IncGeo_BoroughName', 'Is_RepeatedCall', 'Latitude', 'Longitude', 'distance_fire_to_station', 'NumOfCalls_bucket', 'Is_central_London']
Train: 310660 | Test: 116221 | Val: 137825


In [22]:
#UNCHANGED
# Encoding configuration
feature_config = {
    "Month":            {"encoding": "CYCLIC"},
    "Weekday":          {"encoding": "CYCLIC"},
    "Hour":             {"encoding": "CYCLIC"},
    "CalYear":          {"encoding": "NUMERIC_KEEP"},
    "Is_Nightshift":    {"encoding": "BINARY_KEEP"},
    "Is_Rush_Hour":     {"encoding": "BINARY_KEEP"},
    "Is_Weekend":       {"encoding": "BINARY_KEEP"},
    "Is_Public_Holiday":{"encoding": "BINARY_KEEP"},
    "Is_SpecialService":{"encoding": "BINARY_KEEP"},
    "Is_RepeatedCall":  {"encoding": "BINARY_KEEP"},
    "Is_central_London":{"encoding": "BINARY_KEEP"},
    "Latitude":         {"encoding": "NUMERIC_KEEP"},
    "Longitude":        {"encoding": "NUMERIC_KEEP"},
    "distance_fire_to_station": {"encoding": "NUMERIC_KEEP"},
    "IncidentGroup":    {"encoding": "ONE_HOT"},
    "PropertyCategory": {"encoding": "ONE_HOT"},
    "NumOfCalls_bucket":{"encoding": "ONE_HOT"},
    "SpecialServiceType":{"encoding": "TOP_N_PLUS_ONE_HOT", "top_n": 10},
    "DeployedFromStation_Name": {"encoding": "LEAVE_ONE_OUT_TARGET"},
    "PropertyType":     {"encoding": "LEAVE_ONE_OUT_TARGET"},
    "IncGeo_BoroughName":{"encoding": "LEAVE_ONE_OUT_TARGET"}
}

DROP_COLS_ENCODING = ["PumpOrder"]

def apply_encoding(X, y, config, is_train=True, fitted_encoders=None):
    if is_train:
        fitted_encoders = {}
    
    encoded_parts = []
    
    for col, cfg in config.items():
        if col not in X.columns:
            continue
        
        method = cfg["encoding"]
        
        if method in ["NUMERIC_KEEP", "BINARY_KEEP"]:
            encoded_parts.append(X[[col]])
            
        elif method == "CYCLIC":
            periods = {"Month": 12, "Weekday": 7, "Hour": 24}
            p = periods[col]
            encoded_parts.append(pd.DataFrame({
                f"{col}_sin": np.sin(2 * np.pi * X[col] / p),
                f"{col}_cos": np.cos(2 * np.pi * X[col] / p)
            }, index=X.index))
            
        elif method in ["ONE_HOT", "TOP_N_PLUS_ONE_HOT"]:
            if is_train:
                data_to_fit = X[[col]].copy()
                top_n_list = None
                if method == "TOP_N_PLUS_ONE_HOT":
                    top_n_list = X[col].value_counts().head(cfg.get("top_n", 10)).index.tolist()
                    data_to_fit[col] = data_to_fit[col].where(data_to_fit[col].isin(top_n_list), "Other")
                enc = ce.OneHotEncoder(cols=[col], use_cat_names=True).fit(data_to_fit)
                fitted_encoders[col] = (enc, top_n_list)
            
            enc, top_n_list = fitted_encoders[col]
            data_to_transform = X[[col]].copy()
            if top_n_list:
                data_to_transform[col] = data_to_transform[col].where(data_to_transform[col].isin(top_n_list), "Other")
            encoded_parts.append(enc.transform(data_to_transform))

        elif method == "LEAVE_ONE_OUT_TARGET":
            if is_train:
                enc = ce.LeaveOneOutEncoder(cols=[col]).fit(X[[col]], y)
                fitted_encoders[col] = enc
            encoded_parts.append(fitted_encoders[col].transform(X[[col]]))
            
    X_encoded = pd.concat(encoded_parts, axis=1)
    return X_encoded, fitted_encoders

# Run encoding — fit only on train, transform on all three
X_train_final, encoders = apply_encoding(X_train, y_train_log, feature_config, is_train=True)
X_test_final,  _        = apply_encoding(X_test,  None, feature_config, is_train=False, fitted_encoders=encoders)
# CHANGED: added validation set encoding
X_val_final,   _        = apply_encoding(X_val,   None, feature_config, is_train=False, fitted_encoders=encoders)

In [23]:
#CHANGED
# CHANGE: scaler now applied to validation set as well
scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_final),
    columns=X_train_final.columns,
    index=X_train_final.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test_final),
    columns=X_test_final.columns,
    index=X_test_final.index
)
# CHANGED: added validation set scaling
X_val_scaled = pd.DataFrame(
    scaler.transform(X_val_final),
    columns=X_val_final.columns,
    index=X_val_final.index
)

In [24]:
#CHANGED
# CHANGE: added saving of validation set CSVs
X_train_scaled.to_csv(data_path / "X_train_final.csv", index=False)
X_test_scaled.to_csv(data_path / "X_test_final.csv",  index=False)
X_val_scaled.to_csv(data_path / "X_val_final.csv",    index=False)   # CHANGED: new

y_train.to_csv(data_path / "y_train.csv",     index=False)
y_test.to_csv(data_path / "y_test.csv",       index=False)
y_val.to_csv(data_path / "y_val.csv",         index=False)           # CHANGED: new

y_train_log.to_csv(data_path / "y_train_log.csv", index=False)
y_test_log.to_csv(data_path / "y_test_log.csv",   index=False)
y_val_log.to_csv(data_path / "y_val_log.csv",     index=False)       # CHANGED: new

print("All datasets saved!")

All datasets saved!
